# 14 · Parent Document Retriever

Search small child chunks; return the larger parent for context.

**Analogy handbook:** [parent-doc](../retriever-analogy-handbook.html#parent-doc)  
**Prerequisite:** run `00_basics_concepts.ipynb` once (or the setup cells below) so the Chroma index exists.

### Learning loop
1. Skim the analogy for this technique  
2. Run setup (reuse index if possible)  
3. Run the practical cells  
4. Ask: *Did this fix the failure mode, or only reshuffle noise?*


## Shared setup

These cells install packages, load the Llama 2 PDF, build/load the Chroma index, and define helpers.

> Prefer `REBUILD_INDEX = False` after the first successful build so later method notebooks reuse the same store.


### Learning: !pip install langchain_community langchain_text_splitters langchain_op

**What you'll learn:** Install the packages this notebook needs.

**What this cell does:** Installs required Python packages into the runtime.

**Watch for:** Run once; restart runtime if Colab asks.



In [ ]:
!pip install langchain_community langchain_text_splitters langchain_openai langchain_chroma pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 378.1/378.1 kB 2.2 MB/s eta 0:00:00


### Learning: IMPORTS

**What you'll learn:** Bring in LangChain, embeddings, and vector-store modules.

**What this cell does:** Runs `IMPORTS` and prints intermediate results you can inspect.

**Watch for:** If an import fails, re-run the install cell.



In [ ]:
print("All imports and setup starting...")

# ============================================================
# 1. IMPORTS
# ============================================================

from pathlib import Path
import getpass
import os
import shutil

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma

from langchain_classic.chains.hyde.base import (
    HypotheticalDocumentEmbedder
)

All imports and setup starting...


/tmp/ipykernel_520/2286258816.py:12: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


### Learning: from google.colab import userdata

**What you'll learn:** Bring in LangChain, embeddings, and vector-store modules.

**What this cell does:** Runs `from google.colab import userdata` and prints intermediate results you can inspect.

**Watch for:** If an import fails, re-run the install cell.



In [ ]:
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

### Learning: OPENAI API KEY

**What you'll learn:** Authenticate so embedding and chat calls can run.

**What this cell does:** Runs `OPENAI API KEY` and prints intermediate results you can inspect.

**Watch for:** Never hardcode secrets in shared notebooks.



In [ ]:
# ============================================================
# 2. OPENAI API KEY
# ============================================================

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass(
        "Enter your OpenAI API key: "
    )

print("OpenAI API key configured successfully.")

OpenAI API key configured successfully.


### Learning: DATA DIRECTORY

**What you'll learn:** Locate and load the Llama 2 paper as Document pages.

**What this cell does:** Runs `DATA DIRECTORY` and prints intermediate results you can inspect.

**Watch for:** Confirm page count and first-page text look sane.



In [ ]:
# ============================================================
# 3. DATA DIRECTORY
# ============================================================

DATA_DIR = Path(
    r"/content/"
)

preferred_pdf = DATA_DIR / "llama2-research-paper.pdf"

### Learning: FIND PDF

**What you'll learn:** Locate and load the Llama 2 paper as Document pages.

**What this cell does:** Runs `FIND PDF` and prints intermediate results you can inspect.

**Watch for:** Confirm page count and first-page text look sane.



In [ ]:
# ============================================================
# 4. FIND PDF
# ============================================================

if preferred_pdf.exists():

    PDF_PATH = preferred_pdf

else:

    available_pdfs = sorted(
        DATA_DIR.glob("*.pdf")
    )

    if len(available_pdfs) == 1:

        PDF_PATH = available_pdfs[0]

    elif len(available_pdfs) == 0:

        raise FileNotFoundError(
            f"No PDF file was found inside:\n{DATA_DIR}"
        )

    else:

        raise RuntimeError(
            "Multiple PDF files were found. "
            "Please set PDF_PATH manually.\n"
            + "\n".join(
                str(path)
                for path in available_pdfs
            )
        )


print("PDF found:")
print(PDF_PATH)

PDF found:
/content/llama2-research-paper.pdf


### Learning: LOAD PDF

**What you'll learn:** Locate and load the Llama 2 paper as Document pages.

**What this cell does:** Runs `LOAD PDF` and prints intermediate results you can inspect.

**Watch for:** Confirm page count and first-page text look sane.



In [15]:
# ============================================================
# 5. LOAD PDF
# ============================================================

loader = PyPDFLoader(
    str(PDF_PATH)
)

pages = loader.load()

print(
    f"\nTotal PDF pages loaded: {len(pages)}"
)


# ============================================================
# 6. INSPECT FIRST PAGE
# ============================================================

print("\nFirst-page metadata:")
print(
    pages[0].metadata
)

print("\nFirst 1,000 characters:")
print(
    pages[0].page_content[:1000]
)


Total PDF pages loaded: 77

First-page metadata:
{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '/content/llama2-research-paper.pdf', 'total_pages': 77, 'page': 0, 'page_label': '1'}

First 1,000 characters:
Llama 2: Open Foundation and Fine-Tuned Chat Models
Hugo Touvron∗ Louis Martin† Kevin Stone†
Peter Albert Amjad Almahairi Yasmine Babaei Nikolay Bashlykov Soumya Batra
Prajjwal Bhargava Shruti Bhosale Dan Bikel Lukas Blecher Cristian Canton Ferrer Moya Chen
Guillem Cucurull David Esiobu Jude Fernandes Jeremy Fu Wenyin Fu Brian Fuller
Cynthia Gao Vedanuj Goswami Naman Goyal Anthony Hartshorn Saghar Hosseini Rui Hou
Hakan Inan Marcin Kardas Viktor Kerkez Madian Khabsa Isabel Kloumann Art

### Learning: IDENTIFY PAPER SECTIONS

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** Defines helper logic for: IDENTIFY PAPER SECTIONS.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [ ]:
# ============================================================
# 7. IDENTIFY PAPER SECTIONS
# ============================================================

def identify_section(
    paper_page: int
) -> str:

    if 1 <= paper_page <= 2:
        return "front_matter"

    if 3 <= paper_page <= 4:
        return "introduction"

    if 5 <= paper_page <= 7:
        return "pretraining"

    if 8 <= paper_page <= 19:
        return "fine_tuning"

    if 20 <= paper_page <= 31:
        return "safety"

    if 32 <= paper_page <= 35:
        return "discussion"

    if paper_page == 36:
        return "conclusion"

    if 37 <= paper_page <= 45:
        return "references"

    if 46 <= paper_page <= 77:
        return "appendix"

    return "unknown"


# ============================================================
# 8. ADD METADATA
# ============================================================

for page_document in pages:

    page_index = int(
        page_document.metadata.get(
            "page",
            0
        )
    )

    paper_page = (
        page_index + 1
    )

    page_document.metadata.update(
        {
            "paper": "Llama 2",
            "organization": "Meta",
            "year": 2023,
            "document_type": "research_paper",
            "paper_page": paper_page,
            "section": identify_section(
                paper_page
            ),
            "access_level": "public",
        }
    )


print("\nMetadata after enrichment:")

for page_document in pages[:5]:

    print(
        page_document.metadata
    )



Metadata after enrichment:
{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'D:\\complete_content_new\\Full-Stack-GenAI-Bootcamp-1.0\\Class-37-08-Aug-2026-prompting\\data\\llama2-research-paper.pdf', 'total_pages': 77, 'page': 0, 'page_label': '1', 'paper': 'Llama 2', 'organization': 'Meta', 'year': 2023, 'document_type': 'research_paper', 'paper_page': 1, 'section': 'front_matter', 'access_level': 'public'}
{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5',

### Learning: TEXT SPLITTING

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** Runs `TEXT SPLITTING` and prints intermediate results you can inspect.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [ ]:
# ============================================================
# 9. TEXT SPLITTING
# ============================================================

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True,
)

chunks = text_splitter.split_documents(
    pages
)

print(
    f"\nTotal pages: {len(pages)}"
)

print(
    f"Total chunks: {len(chunks)}"
)


# ============================================================
# 10. ADD CHUNK IDs
# ============================================================

for chunk_number, chunk in enumerate(
    chunks
):

    paper_page = chunk.metadata.get(
        "paper_page",
        "unknown"
    )

    chunk.metadata[
        "chunk_id"
    ] = (
        f"llama2-page-"
        f"{paper_page}-"
        f"chunk-{chunk_number}"
    )


print("\nFirst chunk content:")

print(
    chunks[0].page_content[:1000]
)

print("\nFirst chunk metadata:")

print(
    chunks[0].metadata
)


Total pages: 77
Total chunks: 343

First chunk content:
Llama 2: Open Foundation and Fine-Tuned Chat Models
Hugo Touvron∗ Louis Martin† Kevin Stone†
Peter Albert Amjad Almahairi Yasmine Babaei Nikolay Bashlykov Soumya Batra
Prajjwal Bhargava Shruti Bhosale Dan Bikel Lukas Blecher Cristian Canton Ferrer Moya Chen
Guillem Cucurull David Esiobu Jude Fernandes Jeremy Fu Wenyin Fu Brian Fuller
Cynthia Gao Vedanuj Goswami Naman Goyal Anthony Hartshorn Saghar Hosseini Rui Hou
Hakan Inan Marcin Kardas Viktor Kerkez Madian Khabsa Isabel Kloumann Artem Korenev
Punit Singh Koura Marie-Anne Lachaux Thibaut Lavril Jenya Lee Diana Liskovich
Yinghai Lu Yuning Mao Xavier Martinet Todor Mihaylov Pushkar Mishra
Igor Molybog Yixin Nie Andrew Poulton Jeremy Reizenstein Rashi Rungta Kalyan Saladi
Alan Schelten Ruan Silva Eric Michael Smith Ranjan Subramanian Xiaoqing Ellen Tan Binh Tang
Ross Taylor Adina Williams Jian Xiang Kuan Puxin Xu Zheng Yan Iliyan Zarov Yuchen Zhang
Angela Fan Melanie Kambadur Shar

### Learning: CREATE EMBEDDING MODEL

**What you'll learn:** Authenticate so embedding and chat calls can run.

**What this cell does:** Runs `CREATE EMBEDDING MODEL` and prints intermediate results you can inspect.

**Watch for:** Never hardcode secrets in shared notebooks.



In [ ]:
# ============================================================
# 11. CREATE EMBEDDING MODEL
# ============================================================

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)


# ============================================================
# 12. TEST EMBEDDING MODEL
# ============================================================

test_vector = embeddings.embed_query(
    "What is Llama 2?"
)

print(
    f"\nEmbedding dimensions: "
    f"{len(test_vector)}"
)

print(
    f"First 10 values: "
    f"{test_vector[:10]}"
)



Embedding dimensions: 1536
First 10 values: [0.0027942657470703125, -0.0521240234375, -0.021087646484375, -0.055419921875, -0.026397705078125, 0.028961181640625, -0.002071380615234375, 0.034759521484375, -0.0164794921875, -0.0245208740234375]


### Learning: CHROMA CONFIGURATION

**What you'll learn:** Build or reload the vector index used by retrievers.

**What this cell does:** Runs `CHROMA CONFIGURATION` and prints intermediate results you can inspect.

**Watch for:** Use REBUILD_INDEX=False after the first successful build.



In [ ]:
# ============================================================
# 13. CHROMA CONFIGURATION
# ============================================================

PERSIST_DIRECTORY = (
    DATA_DIR
    / "chroma_llama2_retriever"
)

COLLECTION_NAME = (
    "llama2_retriever_demo"
)


### Learning: CREATE OR LOAD VECTOR STORE

**What you'll learn:** Build or reload the vector index used by retrievers.

**What this cell does:** Runs `CREATE OR LOAD VECTOR STORE` and prints intermediate results you can inspect.

**Watch for:** Use REBUILD_INDEX=False after the first successful build.



In [ ]:
# ============================================================
# 14. CREATE OR LOAD VECTOR STORE
# ============================================================

# True  = rebuild complete vector DB
# False = reuse existing vector DB

REBUILD_INDEX = True

### Learning: VERIFY VECTOR STORE

**What you'll learn:** Break pages into retrieval-sized chunks.

**What this cell does:** Runs `VERIFY VECTOR STORE` and prints intermediate results you can inspect.

**Watch for:** Chunk size trades precision vs context — inspect a sample.



In [ ]:
if REBUILD_INDEX:

    print(
        "\nRebuilding vector store..."
    )

    if PERSIST_DIRECTORY.exists():

        shutil.rmtree(
            PERSIST_DIRECTORY,
            ignore_errors=True
        )

    vector_store = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        collection_name=COLLECTION_NAME,
        persist_directory=str(
            PERSIST_DIRECTORY
        ),
        collection_configuration={
            "hnsw": {
                "space": "cosine"
            }
        },
    )

    print(
        "New vector store created."
    )

else:

    if not PERSIST_DIRECTORY.exists():

        print(
            "\nExisting vector DB "
            "not found."
        )

        print(
            "Creating a new vector store..."
        )

        vector_store = Chroma.from_documents(
            documents=chunks,
            embedding=embeddings,
            collection_name=COLLECTION_NAME,
            persist_directory=str(
                PERSIST_DIRECTORY
            ),
            collection_configuration={
                "hnsw": {
                    "space": "cosine"
                }
            },
        )

        print(
            "New vector store created."
        )

    else:

        print(
            "\nLoading existing "
            "vector store..."
        )

        vector_store = Chroma(
            collection_name=COLLECTION_NAME,
            embedding_function=embeddings,
            persist_directory=str(
                PERSIST_DIRECTORY
            ),
        )

        print(
            "Existing vector store "
            "loaded."
        )


# ============================================================
# 15. VERIFY VECTOR STORE
# ============================================================

stored_count = (
    vector_store
    ._collection
    .count()
)

print(
    f"\nStored chunks: "
    f"{stored_count}"
)

print(
    f"Persisted at: "
    f"{PERSIST_DIRECTORY}"
)



Rebuilding vector store...
New vector store created.

Stored chunks: 343
Persisted at: D:\complete_content_new\Full-Stack-GenAI-Bootcamp-1.0\Class-37-08-Aug-2026-prompting\data\chroma_llama2_retriever


### Learning: parent document retriever practical

**What you'll learn:** Search small child chunks, return larger parent context.

**What this cell does:** Runs `parent document retriever practical` and prints intermediate results you can inspect.

**Watch for:** Child finds; parent explains — don't confuse the two stores.



In [ ]:
# parent document retriever practical

### Learning: PARENT DOCUMENT RETRIEVER PRACTICAL

**What you'll learn:** Bring in LangChain, embeddings, and vector-store modules.

**What this cell does:** Runs `PARENT DOCUMENT RETRIEVER PRACTICAL` and prints intermediate results you can inspect.

**Watch for:** If an import fails, re-run the install cell.



In [ ]:
# ============================================================
# PARENT DOCUMENT RETRIEVER PRACTICAL
# ============================================================

# ------------------------------------------------------------
# 1. IMPORTS
# ------------------------------------------------------------

from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.stores import InMemoryStore

### Learning: CONFIGURATION

**What you'll learn:** Build or reload the vector index used by retrievers.

**What this cell does:** Runs `CONFIGURATION` and prints intermediate results you can inspect.

**Watch for:** Use REBUILD_INDEX=False after the first successful build.



In [ ]:

# ============================================================
# 2. CONFIGURATION
# ============================================================

PARENT_COLLECTION_NAME = "parent_document_retriever_demo"

PARENT_VECTORSTORE_DIR = (
    DATA_DIR / "parent_document_chroma"
)


### Learning: CREATE PARENT SPLITTER

**What you'll learn:** Break pages into retrieval-sized chunks.

**What this cell does:** Runs `CREATE PARENT SPLITTER` and prints intermediate results you can inspect.

**Watch for:** Chunk size trades precision vs context — inspect a sample.



In [ ]:
# ============================================================
# 3. CREATE PARENT SPLITTER
# ============================================================

# Large chunks that will finally be returned to the user/LLM

parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=200
)



### Learning: CREATE CHILD SPLITTER

**What you'll learn:** Break pages into retrieval-sized chunks.

**What this cell does:** Runs `CREATE CHILD SPLITTER` and prints intermediate results you can inspect.

**Watch for:** Chunk size trades precision vs context — inspect a sample.



In [ ]:
# ============================================================
# 4. CREATE CHILD SPLITTER
# ============================================================

# Small chunks used for embedding and similarity search

child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=50
)


print("Parent and child splitters created.")


Parent and child splitters created.


### Learning: CREATE EMPTY VECTOR STORE

**What you'll learn:** Break pages into retrieval-sized chunks.

**What this cell does:** Runs `CREATE EMPTY VECTOR STORE` and prints intermediate results you can inspect.

**Watch for:** Chunk size trades precision vs context — inspect a sample.



In [ ]:
# ============================================================
# 5. CREATE EMPTY VECTOR STORE
# ============================================================

# IMPORTANT:
# Child chunks will be stored here.

parent_vector_store = Chroma(
    collection_name=PARENT_COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=str(
        PARENT_VECTORSTORE_DIR
    )
)


print("Child vector store created.")


Child vector store created.


### Learning: CREATE DOCUMENT STORE

**What you'll learn:** Search small child chunks, return larger parent context.

**What this cell does:** Runs `CREATE DOCUMENT STORE` and prints intermediate results you can inspect.

**Watch for:** Child finds; parent explains — don't confuse the two stores.



In [ ]:
# ============================================================
# 6. CREATE DOCUMENT STORE
# ============================================================

# IMPORTANT:
# Parent documents are stored separately here.

docstore = InMemoryStore()


print("Parent document store created.")

Parent document store created.


### Learning: CREATE PARENT DOCUMENT RETRIEVER

**What you'll learn:** Break pages into retrieval-sized chunks.

**What this cell does:** Runs `CREATE PARENT DOCUMENT RETRIEVER` and prints intermediate results you can inspect.

**Watch for:** Chunk size trades precision vs context — inspect a sample.



In [ ]:
# ============================================================
# 7. CREATE PARENT DOCUMENT RETRIEVER
# ============================================================

parent_retriever = ParentDocumentRetriever(
    vectorstore=parent_vector_store,
    docstore=docstore,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)


print(
    "ParentDocumentRetriever created successfully."
)



ParentDocumentRetriever created successfully.


### Learning: ADD ORIGINAL DOCUMENTS

**What you'll learn:** Locate and load the Llama 2 paper as Document pages.

**What this cell does:** Runs `ADD ORIGINAL DOCUMENTS` and prints intermediate results you can inspect.

**Watch for:** Confirm page count and first-page text look sane.



In [ ]:
# ============================================================
# 8. ADD ORIGINAL DOCUMENTS
# ============================================================

# "pages" comes from your previous PyPDFLoader code.
#
# Internally:
#
# Original Pages
#      ↓
# Parent Splitter
#      ↓
# Large Parent Chunks
#      ↓
# Child Splitter
#      ↓
# Small Child Chunks
#
# Child Chunks  → Vector Store
# Parent Chunks → Docstore

parent_retriever.add_documents(
    pages
)


print(
    "Documents added to ParentDocumentRetriever."
)

Documents added to ParentDocumentRetriever.


### Learning: CHECK CHILD CHUNK COUNT

**What you'll learn:** Break pages into retrieval-sized chunks.

**What this cell does:** Runs `CHECK CHILD CHUNK COUNT` and prints intermediate results you can inspect.

**Watch for:** Chunk size trades precision vs context — inspect a sample.



In [ ]:
# ============================================================
# 9. CHECK CHILD CHUNK COUNT
# ============================================================

child_count = (
    parent_vector_store
    ._collection
    .count()
)

print(
    f"Total child chunks stored in vector DB: "
    f"{child_count}"
)


Total child chunks stored in vector DB: 856


### Learning: USER QUERY

**What you'll learn:** Execute the next step in the retrieval pipeline and observe the output.

**What this cell does:** Runs `USER QUERY` and prints intermediate results you can inspect.

**Watch for:** Relate this step to find → order → trim in the RAG pipeline.



In [ ]:
# ============================================================
# 10. USER QUERY
# ============================================================

query = (
    "How was Llama 2 trained using human feedback?"
)


print("\nUSER QUERY:")
print(query)




USER QUERY:
How was Llama 2 trained using human feedback?


### Learning: RETRIEVE PARENT DOCUMENTS

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** Executes retrieval/generation for: RETRIEVE PARENT DOCUMENTS.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [ ]:
# ============================================================
# 11. RETRIEVE PARENT DOCUMENTS
# ============================================================

parent_documents = (
    parent_retriever.invoke(
        query
    )
)


print(
    "\nPARENT DOCUMENT RETRIEVAL RESULTS"
)

print(
    "=" * 100
)


for i, document in enumerate(
    parent_documents,
    start=1
):

    print(
        f"\nRESULT {i}"
    )

    print(
        "Paper page:",
        document.metadata.get(
            "paper_page"
        )
    )

    print(
        "Section:",
        document.metadata.get(
            "section"
        )
    )

    print(
        "Returned document length:",
        len(document.page_content)
    )

    print(
        "-" * 100
    )

    print(
        document.page_content[:1500]
    )



PARENT DOCUMENT RETRIEVAL RESULTS

RESULT 1
Paper page: 5
Section: pretraining
Returned document length: 1923
----------------------------------------------------------------------------------------------------
Figure 4: Training ofLlama 2-Chat: This process begins with thepretraining of Llama 2 using publicly
available online sources. Following this, we create an initial version ofLlama 2-Chatthrough the application
of supervised fine-tuning. Subsequently, the model is iteratively refined using Reinforcement Learning
with Human Feedback(RLHF) methodologies, specifically through rejection sampling and Proximal Policy
Optimization (PPO). Throughout the RLHF stage, the accumulation ofiterative reward modeling datain
parallel with model enhancements is crucial to ensure the reward models remain within distribution.
2 Pretraining
Tocreatethenewfamilyof Llama 2models,webeganwiththepretrainingapproachdescribedinTouvronetal.
(2023), using an optimized auto-regressive transformer, but made se

### Learning: DIRECT CHILD-CHUNK SEARCH

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** Executes retrieval/generation for: DIRECT CHILD-CHUNK SEARCH.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [ ]:
# ============================================================
# 12. DIRECT CHILD-CHUNK SEARCH
# ============================================================

# This directly searches the underlying vector store.
# These are the SMALL chunks that actually match the query.

child_documents = (
    parent_vector_store
    .similarity_search(
        query=query,
        k=4
    )
)


print(
    "\n\nDIRECT CHILD-CHUNK SEARCH"
)

print(
    "=" * 100
)


for i, document in enumerate(
    child_documents,
    start=1
):

    print(
        f"\nCHILD RESULT {i}"
    )

    print(
        "Child length:",
        len(document.page_content)
    )

    print(
        "Parent ID:",
        document.metadata.get(
            "doc_id"
        )
    )

    print(
        "-" * 100
    )

    print(
        document.page_content
    )



DIRECT CHILD-CHUNK SEARCH

CHILD RESULT 1
Child length: 312
Parent ID: 0541ba9d-32db-498f-acd8-c9279e66c499
----------------------------------------------------------------------------------------------------
Figure 4: Training ofLlama 2-Chat: This process begins with thepretraining of Llama 2 using publicly
available online sources. Following this, we create an initial version ofLlama 2-Chatthrough the application
of supervised fine-tuning. Subsequently, the model is iteratively refined using Reinforcement Learning

CHILD RESULT 2
Child length: 313
Parent ID: e81d3afc-48b2-4ab0-96b4-4cd51be89832
----------------------------------------------------------------------------------------------------
reward models improved, and we were able to train progressively better versions forLlama 2-Chat (see
the results in Section 5, Figure 20).Llama 2-Chat improvement also shifted the model’s data distribution.
Since reward model accuracy can quickly degrade if not exposed to this new sample dist

### Learning: COMPARE CHILD VS PARENT

**What you'll learn:** Search small child chunks, return larger parent context.

**What this cell does:** Runs `COMPARE CHILD VS PARENT` and prints intermediate results you can inspect.

**Watch for:** Child finds; parent explains — don't confuse the two stores.



In [ ]:
# ============================================================
# 13. COMPARE CHILD VS PARENT
# ============================================================

print(
    "\n\nCHILD VS PARENT COMPARISON"
)

print(
    "=" * 100
)


print("\nCHILD RESULTS:")

for i, document in enumerate(
    child_documents,
    start=1
):

    print(
        f"{i}. Length = "
        f"{len(document.page_content)}"
    )


print("\nPARENT RESULTS:")

for i, document in enumerate(
    parent_documents,
    start=1
):

    print(
        f"{i}. Length = "
        f"{len(document.page_content)}"
    )




CHILD VS PARENT COMPARISON

CHILD RESULTS:
1. Length = 312
2. Length = 313
3. Length = 334
4. Length = 316

PARENT RESULTS:
1. Length = 1923
2. Length = 1999
3. Length = 1255


### Learning: INSPECT CHILD → PARENT RELATIONSHIP

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** Runs `INSPECT CHILD → PARENT RELATIONSHIP` and prints intermediate results you can inspect.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [ ]:
# ============================================================
# 14. INSPECT CHILD → PARENT RELATIONSHIP
# ============================================================

print(
    "\n\nCHILD TO PARENT IDs"
)

print(
    "=" * 100
)


for i, child in enumerate(
    child_documents,
    start=1
):

    parent_id = child.metadata.get(
        "doc_id"
    )

    print(
        f"\nChild {i}"
    )

    print(
        "Parent ID:",
        parent_id
    )

    print(
        "Child content:"
    )

    print(
        child.page_content[:300]
    )





CHILD TO PARENT IDs

Child 1
Parent ID: 0541ba9d-32db-498f-acd8-c9279e66c499
Child content:
Figure 4: Training ofLlama 2-Chat: This process begins with thepretraining of Llama 2 using publicly
available online sources. Following this, we create an initial version ofLlama 2-Chatthrough the application
of supervised fine-tuning. Subsequently, the model is iteratively refined using Reinforcem

Child 2
Parent ID: e81d3afc-48b2-4ab0-96b4-4cd51be89832
Child content:
reward models improved, and we were able to train progressively better versions forLlama 2-Chat (see
the results in Section 5, Figure 20).Llama 2-Chat improvement also shifted the model’s data distribution.
Since reward model accuracy can quickly degrade if not exposed to this new sample distributio

Child 3
Parent ID: 87bf0868-82ca-403b-a00c-c4d7dfdecce5
Child content:
3 Fine-tuning
Llama 2-Chat is the result of several months of research and iterative applications of alignment techniques,
including both instruction tuning and

### Learning: FETCH A PARENT DIRECTLY FROM DOCSTORE

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** Runs `FETCH A PARENT DIRECTLY FROM DOCSTORE` and prints intermediate results you can inspect.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [ ]:
# ============================================================
# 15. FETCH A PARENT DIRECTLY FROM DOCSTORE
# ============================================================

# Take the parent ID of the first matching child.

if child_documents:

    parent_id = (
        child_documents[0]
        .metadata
        .get("doc_id")
    )

    if parent_id:

        stored_parent = (
            docstore.mget(
                [parent_id]
            )[0]
        )

        print(
            "\n\nPARENT FETCHED DIRECTLY FROM DOCSTORE"
        )

        print(
            "=" * 100
        )

        print(
            "Parent ID:",
            parent_id
        )

        print(
            "\nParent length:",
            len(
                stored_parent.page_content
            )
        )

        print(
            "\nParent content:"
        )

        print(
            stored_parent.page_content[:2000]
        )



PARENT FETCHED DIRECTLY FROM DOCSTORE
Parent ID: 0541ba9d-32db-498f-acd8-c9279e66c499

Parent length: 1923

Parent content:
Figure 4: Training ofLlama 2-Chat: This process begins with thepretraining of Llama 2 using publicly
available online sources. Following this, we create an initial version ofLlama 2-Chatthrough the application
of supervised fine-tuning. Subsequently, the model is iteratively refined using Reinforcement Learning
with Human Feedback(RLHF) methodologies, specifically through rejection sampling and Proximal Policy
Optimization (PPO). Throughout the RLHF stage, the accumulation ofiterative reward modeling datain
parallel with model enhancements is crucial to ensure the reward models remain within distribution.
2 Pretraining
Tocreatethenewfamilyof Llama 2models,webeganwiththepretrainingapproachdescribedinTouvronetal.
(2023), using an optimized auto-regressive transformer, but made several changes to improve performance.
Specifically, we performed more robust data clea

### Learning: CREATE REUSABLE FUNCTION

**What you'll learn:** Break pages into retrieval-sized chunks.

**What this cell does:** Defines helper logic for: CREATE REUSABLE FUNCTION.

**Watch for:** Chunk size trades precision vs context — inspect a sample.



In [ ]:
# ============================================================
# 16. CREATE REUSABLE FUNCTION
# ============================================================

def parent_document_search(
    query: str
):
    """
    Search small child chunks,
    but return their larger parent chunks.
    """

    documents = (
        parent_retriever.invoke(
            query
        )
    )

    return documents

### Learning: TEST REUSABLE FUNCTION

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** Runs `TEST REUSABLE FUNCTION` and prints intermediate results you can inspect.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [ ]:
# ============================================================
# 17. TEST REUSABLE FUNCTION
# ============================================================

query = (
    "What safety techniques were "
    "used for Llama 2-Chat?"
)

results = parent_document_search(
    query
)


print(
    "\n\nREUSABLE PARENT DOCUMENT RETRIEVER"
)

print(
    "=" * 100
)


for i, document in enumerate(
    results,
    start=1
):

    print(
        f"\nRESULT {i}"
    )

    print(
        "Page:",
        document.metadata.get(
            "paper_page"
        )
    )

    print(
        "Section:",
        document.metadata.get(
            "section"
        )
    )

    print(
        "Length:",
        len(document.page_content)
    )

    print(
        document.page_content[:1200]
    )





REUSABLE PARENT DOCUMENT RETRIEVER

RESULT 1
Page: 24
Section: safety
Length: 1932
advice). The attack vectors explored consist of psychological manipulation (e.g., authority manipulation),
logic manipulation (e.g., false premises), syntactic manipulation (e.g., misspelling), semantic manipulation
(e.g., metaphor), perspective manipulation (e.g., role playing), non-English languages, and others.
Wethendefinebestpracticesforsafeandhelpfulmodelresponses: themodelshouldfirstaddressimmediate
safetyconcernsifapplicable,thenaddressthepromptbyexplainingthepotentialriskstotheuser,andfinally
provide additional information if possible. We also ask the annotators to avoid negative user experience
categories (see Appendix A.5.2). The guidelines are meant to be a general guide for the model and are
iteratively refined and revised to include newly identified risks.
4.2.2 Safety Supervised Fine-Tuning
In accordance with the established guidelines from Section 4.2.1, we gather prompts and demonstrat

### Learning: FINAL CONCEPTUAL FLOW

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** Runs `FINAL CONCEPTUAL FLOW` and prints intermediate results you can inspect.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [ ]:
# ============================================================
# 18. FINAL CONCEPTUAL FLOW
# ============================================================

"""
PARENT DOCUMENT RETRIEVAL

Original Document
        ↓
Parent Splitter
        ↓
Large Parent Chunks
        ↓
Child Splitter
        ↓
Small Child Chunks
        ↓
Create Embeddings for CHILD chunks
        ↓
Store CHILD chunks in Vector DB
        ↓
Store PARENT chunks in Docstore
        ↓

User Query
        ↓
Query Embedding
        ↓
Search CHILD chunks
        ↓
Best child chunk found
        ↓
Read parent ID from child metadata
        ↓
Fetch corresponding PARENT from Docstore
        ↓
Return larger parent context
        ↓
LLM
"""


print(
    "\nParent Document Retriever "
    "practical completed successfully."
)


Parent Document Retriever practical completed successfully.


### Learning: parent_retriever = ParentDocumentRetriever(

**What you'll learn:** Break pages into retrieval-sized chunks.

**What this cell does:** Executes retrieval/generation for: parent_retriever = ParentDocumentRetriever(.

**Watch for:** Chunk size trades precision vs context — inspect a sample.



In [ ]:
parent_retriever = ParentDocumentRetriever(
    vectorstore=parent_vector_store,
    docstore=docstore,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

parent_retriever.add_documents(pages)

documents = parent_retriever.invoke(
    "How was Llama 2 trained using human feedback?"
)

### Learning: Original Document

**What you'll learn:** Break pages into retrieval-sized chunks.

**What this cell does:** Runs `Original Document` and prints intermediate results you can inspect.

**Watch for:** Chunk size trades precision vs context — inspect a sample.



In [ ]:
# Original Document
#         ↓
# Large Parent Chunk
#         ↓
# Small Child Chunks
#         ↓
# Child Embeddings
#         ↓
# Vector Search
#         ↓
# Matched Child
#         ↓
# Parent ID
#         ↓
# Parent Document
#         ↓
# Final Return

### Learning: Child chunks  → Vector DB

**What you'll learn:** Break pages into retrieval-sized chunks.

**What this cell does:** Runs `Child chunks  → Vector DB` and prints intermediate results you can inspect.

**Watch for:** Chunk size trades precision vs context — inspect a sample.



In [ ]:
Child chunks  → Vector DB
Parent chunks → Docstore

Search Child
Return Parent